## Session 2 · Topic 7 — Combining Tables: Multiple Sheets and Merge (Python)

**Dataset:** `contoso_aus.xlsx` — the **same workbook you used in the Excel case
study** (Contoso Australia). It has five sheets: `Sales`, `Product`, `Customer`,
`Store` and `Date`.

In Excel you worked across these sheets by hand: on **Day 1** you filled in the
calculated columns (revenue, cost, margin), and on **Day 2** you used **XLOOKUP**
to pull each product's name and category into the `Sales` sheet. This notebook
does the same two jobs in pandas:

1. **Read more than one sheet** into separate DataFrames.
2. **Rebuild a few Day 1 calculated columns** as vectorised operations.
3. **Merge** `Sales` with `Product` — pandas' version of a **SQL join**, and the
   direct equivalent of Day 2's XLOOKUP.

Replace every `# TODO` with your own code and run the cell.

### 1. Read more than one table

`pd.read_excel()` reads **one sheet at a time** — pass `sheet_name=` to choose
which. Read the `Sales` sheet into `sales` and the `Product` sheet into
`product`, then print the shape of each.

> The `Sales` sheet has empty columns (`ProductName`, `LineRevenue`, `Margin`, …)
> — those are the ones you filled *by hand* in Excel. Here you'll rebuild them,
> so ignore them for now.

In [1]:
import pandas as pd
from pathlib import Path

DATA_FOLDER = Path("../data")
DATA_FILE = DATA_FOLDER / "contoso_aus.xlsx"

df_sales = pd.read_excel(DATA_FILE, sheet_name="Sales")
df_product = pd.read_excel(DATA_FILE, sheet_name="Product")

print("sales  :", df_sales.shape)
print("product:", df_product.shape)

sales  : (7739, 23)
product: (1951, 14)


### 2. Keep the base columns of Sales

Reduce `sales` to the columns that actually hold data — the ones every order line
comes with. Overwrite `sales` with just these:

`OrderKey`, `LineNumber`, `OrderDate`, `ProductKey`, `CustomerKey`, `StoreKey`,
`Quantity`, `UnitPrice`, `NetPrice`, `UnitCost`

Print `sales.head()`.

In [2]:
base_cols = ["OrderKey", "LineNumber", "OrderDate", "ProductKey",
             "CustomerKey", "StoreKey", "Quantity", "UnitPrice",
             "NetPrice", "UnitCost"]
df_sales = df_sales[base_cols]
df_sales.head()

,OrderKey,LineNumber,OrderDate,ProductKey,CustomerKey,StoreKey,Quantity,UnitPrice,NetPrice,UnitCost
0,141006,0,2016-05-20,1315,35688,50,6,328.000,328.00000,150.832
1,148000,0,2016-05-27,1576,166973,60,1,7.794,7.79400,3.972
2,148000,1,2016-05-27,1029,166973,60,3,320.000,275.20000,106.016
3,148006,0,2016-05-27,1790,91603,35,7,30.100,27.39100,15.344
4,148006,1,2016-05-27,158,91603,35,4,879.992,774.39296,404.680


### 3. Rebuild the Day 1 calculated columns (vectorised)

In Excel you wrote one formula and filled it down thousands of rows. In pandas
you write the expression **once** and it applies to the **whole column at once** —
this is a *vectorised* operation, the pandas equivalent of filling a column down.

Create these five columns (same formulas as the Excel case study):

- `GrossExtension = Quantity * UnitPrice`
- `LineRevenue    = Quantity * NetPrice`
- `LineCost       = Quantity * UnitCost`
- `Margin         = LineRevenue - LineCost`
- `MarginPercent  = Margin / LineRevenue`

Then show `sales[["Quantity", "NetPrice", "LineRevenue", "Margin", "MarginPercent"]].head()`.

In [3]:
df_sales["GrossExtension"] = df_sales["Quantity"] * df_sales["UnitPrice"]
df_sales["LineRevenue"]    = df_sales["Quantity"] * df_sales["NetPrice"]
df_sales["LineCost"]       = df_sales["Quantity"] * df_sales["UnitCost"]
df_sales["Margin"]         = df_sales["LineRevenue"] - df_sales["LineCost"]
df_sales["MarginPercent"]  = df_sales["Margin"] / df_sales["LineRevenue"]

df_sales[["Quantity", "NetPrice", "LineRevenue", "Margin", "MarginPercent"]].head()

,Quantity,NetPrice,LineRevenue,Margin,MarginPercent
0,6,328.00000,1968.00000,1063.00800,0.540146
1,1,7.79400,7.79400,3.82200,0.490377
2,3,275.20000,825.60000,507.55200,0.614767
3,7,27.39100,191.73700,84.32900,0.439816
4,4,774.39296,3097.57184,1478.85184,0.477423


### 4. Connecting tables — a merge is a SQL join

`sales` tells you *what* was sold (a `ProductKey`) but not the product's **name**,
**category** or **brand** — those live in the `product` table. Connecting them is
exactly a **join**, which you've seen in SQL:

```sql
SELECT s.*, p.ProductName, p.CategoryName, p.Brand
FROM Sales s
JOIN Product p ON s.ProductKey = p.ProductKey
```

In pandas the same operation is `merge`:

```
sales.merge(product_lookup, on="ProductKey", how="inner")
```

- **`on="ProductKey"`** — the join key: the column both tables share (the SQL
  `ON` clause).
- **`how=`** — the join type: `inner` (only matching keys, like SQL `INNER JOIN`),
  `left` (keep every row of the left table, like `LEFT JOIN`), `right`, or
  `outer` (like `FULL OUTER JOIN`).
- `product` has **one** row per `ProductKey`; `sales` has **many** rows sharing
  each key — a **many-to-one** join. Every sales row picks up its product's
  details, which is precisely what XLOOKUP did in Excel Day 2.

```
   Sales (many)                     Product (one per key)
   ProductKey ─────────────────────▶ ProductKey, ProductName, CategoryName, Brand
```

First, take just the columns you need from `product` into `product_lookup`:
`ProductKey`, `ProductName`, `CategoryName`, `Brand`. Pulling only what you need
keeps the result tidy (a full merge would drag in all 14 product columns).

In [4]:
df_product_lookup = df_product[["ProductKey", "ProductName", "CategoryName", "Brand"]]
df_product_lookup.head()

,ProductKey,ProductName,CategoryName,Brand
0,1,Contoso 512MB MP3 Player E51 Silver,Audio,Contoso
1,2,Contoso 512MB MP3 Player E51 Blue,Audio,Contoso
2,3,Contoso 1G MP3 Player E100 White,Audio,Contoso
3,4,Contoso 2G MP3 Player E200 Silver,Audio,Contoso
4,5,Contoso 2G MP3 Player E200 Red,Audio,Contoso


### 5. Do the merge

Merge `sales` with `product_lookup` on `ProductKey` using an **inner** join, and
assign the result back to `sales`.

Print the shape before and after. Because every `ProductKey` in `sales` exists in
`product`, and `product` has one row per key, the row count should **stay the
same** (7,739) — you've only added columns, not rows. Then show
`sales[["ProductKey", "ProductName", "CategoryName", "Brand", "LineRevenue"]].head()`.

In [5]:
print("before:", df_sales.shape)
df_sales = df_sales.merge(df_product_lookup, on="ProductKey", how="inner")
print("after :", df_sales.shape)

df_sales[["ProductKey", "ProductName", "CategoryName", "Brand", "LineRevenue"]].head()

before: (7739, 15)
after : (7739, 18)


,ProductKey,ProductName,CategoryName,Brand,LineRevenue
0,1315,Contoso Conversion Lens M550 Pink,Cameras and camcorders,Contoso,1968.00000
1,1576,SV DVD Movies E100 Yellow,"Music, Movies and Audio Books",Southridge Video,7.79400
2,1029,A. Datum Rangefinder Digital Camera X200 Azure,Cameras and camcorders,A. Datum,825.60000
3,1790,MGS Dungeon Siege: Legends of Aranna 2009 E146,Games and Toys,Tailspin Toys,191.73700
4,158,"Adventure Works 37"" 1080p LCD HDTV M150W Black",TV and Video,Adventure Works,3097.57184


### 6. The payoff — a question you couldn't answer before

Before the merge you only had a `ProductKey`; now you have the category. So you
can finally ask a business question: **which category earns the most revenue?**

Group by `CategoryName`, sum `LineRevenue`, and sort highest to lowest.

> This is a preview of Session 3's `groupby()` — here it just shows *why* the
> merge was worth doing.

In [6]:
revenue_by_category = (
    df_sales.groupby("CategoryName")["LineRevenue"].sum().sort_values(ascending=False)
)
revenue_by_category

CategoryName
Computers                        3.437936e+06
Cell phones                      1.179629e+06
Home Appliances                  8.532629e+05
TV and Video                     7.369053e+05
Cameras and camcorders           6.251748e+05
Music, Movies and Audio Books    4.021560e+05
Audio                            2.048698e+05
Games and Toys                   6.118140e+04
Name: LineRevenue, dtype: float64

### 7. Wrap-up

In your own words (2–3 sentences): what does a `merge` do, and which SQL operation
is it the same as? Why did the row count stay at 7,739 after the merge instead of
growing? And how is this merge the same idea as the XLOOKUP you wrote in Excel
Day 2?